## ShortCut Connections

Why we need shortcut connections?

its sloves the vanishing gradient problem.

In [199]:
import torch 
import torch.nn as nn

In [200]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*
                    (x+0.4475*torch.pow(x,3))
                    ))

In [201]:
class exampleDeepNeuralNetworks(nn.Module):
    def __init__(self,Layer,shortcut):
        super().__init__()
        self.shortcut=shortcut
        self.layers=nn.ModuleList([
           nn.Sequential(nn.Linear(Layer[0],Layer[1]),GELU()), #L1
           nn.Sequential(nn.Linear(Layer[1],Layer[2]),GELU()), #L2
           nn.Sequential(nn.Linear(Layer[2],Layer[3]),GELU()), #L3
           nn.Sequential(nn.Linear(Layer[3],Layer[4]),GELU()),
           nn.Sequential(nn.Linear(Layer[4],Layer[5]),GELU()) #L4
        ])

    def forward(self,x):
        for layer in self.layers:
            ##
            layer_out=layer(x)

            if self.shortcut and x.shape == layer_out.shape: ## if shortcut is true
                x=x + layer_out
            else: ## shortcut = off
                x=layer_out
        return x

In [202]:
Layer=[3,3,3,3,3,1]
example_inp=torch.tensor([[-1.,0.0,1]])
torch.manual_seed(123)
dnn_without_shortcut=exampleDeepNeuralNetworks(
    Layer,shortcut=False
)
dnn_without_shortcut

exampleDeepNeuralNetworks(
  (layers): ModuleList(
    (0-3): 4 x Sequential(
      (0): Linear(in_features=3, out_features=3, bias=True)
      (1): GELU()
    )
    (4): Sequential(
      (0): Linear(in_features=3, out_features=1, bias=True)
      (1): GELU()
    )
  )
)

check the gradient 

In [203]:
def print_gradients(model,x):
    # 1. ouptut of the model 
    # model can be with or without shortcut
    # forward pass
    output=model(x)
    target=torch.tensor([[0.]])

   # 2. calculate loss
    loss=nn.MSELoss()
    losses=loss(output,target)

   # 3. backward gradient
    losses.backward()

    for name ,param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")


In [204]:
class exampleDeepNeuralNetworks_4(nn.Module):
    def __init__(self,Layer,shortcut):
        super().__init__()
        self.shortcut=shortcut
        self.layers=nn.ModuleList([
           nn.Sequential(nn.Linear(Layer[0],Layer[1]),GELU()), #L1
           nn.Sequential(nn.Linear(Layer[1],Layer[2]),GELU()), #L2
           nn.Sequential(nn.Linear(Layer[2],Layer[3]),GELU()), #L3
           nn.Sequential(nn.Linear(Layer[3],Layer[4]),GELU()) #L4
        ])

    def forward(self,x):
        for layer in self.layers:
            ##
            layer_out=layer(x)

            if self.shortcut and x.shape == layer_out.shape: ## if shortcut is true
                x=x + layer_out
            else: ## shortcut = off
                x=layer_out
        return x

In [205]:
layer_5_without_shortcut=print_gradients(dnn_without_shortcut,example_inp)

layers.0.0.weight has gradient mean of 0.0002649295493029058
layers.1.0.weight has gradient mean of 8.689835522091016e-05
layers.2.0.weight has gradient mean of 0.0007737756241112947
layers.3.0.weight has gradient mean of 0.0012169942492619157
layers.4.0.weight has gradient mean of 0.00459253741428256


In [206]:
torch.manual_seed(123)
dnn_with_shortcut=exampleDeepNeuralNetworks(
    Layer,shortcut=True
)
layer_5_with_shortcut=print_gradients(dnn_with_shortcut,example_inp)

layers.0.0.weight has gradient mean of 0.04201919212937355
layers.1.0.weight has gradient mean of 0.08804404735565186
layers.2.0.weight has gradient mean of 0.05725563317537308
layers.3.0.weight has gradient mean of 0.03222617134451866
layers.4.0.weight has gradient mean of 0.38649773597717285


In [207]:
torch.manual_seed(123)
Layer=[3,3,3,3,1]
dnn_without_shortcut=exampleDeepNeuralNetworks_4(Layer,shortcut=False)
layer_4_without_shortcut=print_gradients(dnn_without_shortcut,example_inp)


layers.0.0.weight has gradient mean of 0.00026106147561222315
layers.1.0.weight has gradient mean of 0.00016647481243126094
layers.2.0.weight has gradient mean of 0.0008808898855932057
layers.3.0.weight has gradient mean of 0.0020815059542655945


In [208]:
torch.manual_seed(123)
dnn_with_shortcut=exampleDeepNeuralNetworks_4(Layer,shortcut=True)
layer_4_with_shortcut=print_gradients(dnn_with_shortcut,example_inp)

layers.0.0.weight has gradient mean of 0.0034396096598356962
layers.1.0.weight has gradient mean of 0.006220707204192877
layers.2.0.weight has gradient mean of 0.004655098542571068
layers.3.0.weight has gradient mean of 0.02608443796634674


we can see as the DNN layer increase the vanishing GD problem increase .

also effect of shortcut more when the DNN layer increases 

### DNN with 10 layer 

In [209]:
class exampleDeepNeuralNetworks_10(nn.Module):
    def __init__(self,Layer,shortcut):
        super().__init__()
        self.shortcut=shortcut
        self.layers=nn.ModuleList([
           nn.Sequential(nn.Linear(Layer[0],Layer[1]),GELU()), #L1
           nn.Sequential(nn.Linear(Layer[1],Layer[2]),GELU()), #L2
           nn.Sequential(nn.Linear(Layer[2],Layer[3]),GELU()), #L3
           nn.Sequential(nn.Linear(Layer[3],Layer[4]),GELU()), #L4
           nn.Sequential(nn.Linear(Layer[4],Layer[5]),GELU()),
           nn.Sequential(nn.Linear(Layer[5],Layer[6]),GELU()),
           nn.Sequential(nn.Linear(Layer[6],Layer[7]),GELU()),
           nn.Sequential(nn.Linear(Layer[7],Layer[8]),GELU()),
           nn.Sequential(nn.Linear(Layer[8],Layer[9]),GELU()),
           nn.Sequential(nn.Linear(Layer[9],Layer[10]),GELU())
        ])

    def forward(self,x):
        for layer in self.layers:
            ##
            layer_out=layer(x)

            if self.shortcut and x.shape == layer_out.shape: ## if shortcut is true
                x=x + layer_out
            else: ## shortcut = off
                x=layer_out
        return x


In [210]:
torch.manual_seed(123)
Layer=[3,3,3,3,3,3,3,3,3,3,1]
dnn_without_shortcut=exampleDeepNeuralNetworks_10(Layer,shortcut=False)
layer_10_without_shortcut=print_gradients(dnn_without_shortcut,example_inp)

layers.0.0.weight has gradient mean of 3.9404753238159174e-07
layers.1.0.weight has gradient mean of 1.3545056276598189e-07
layers.2.0.weight has gradient mean of 1.3817885928801843e-06
layers.3.0.weight has gradient mean of 2.1010648652008967e-06
layers.4.0.weight has gradient mean of 3.5525097246136284e-06
layers.5.0.weight has gradient mean of 2.6420595531817526e-05
layers.6.0.weight has gradient mean of 0.00017820144421420991
layers.7.0.weight has gradient mean of 7.630451727891341e-05
layers.8.0.weight has gradient mean of 0.001251647132448852
layers.9.0.weight has gradient mean of 0.00676824152469635


In [211]:
torch.manual_seed(123)
dnn_with_shortcut=exampleDeepNeuralNetworks_10(Layer,shortcut=True)
layer_10_with_shortcut=print_gradients(dnn_with_shortcut,example_inp)

layers.0.0.weight has gradient mean of 2.7592508792877197
layers.1.0.weight has gradient mean of 3.7513532638549805
layers.2.0.weight has gradient mean of 2.0684690475463867
layers.3.0.weight has gradient mean of 1.8869272470474243
layers.4.0.weight has gradient mean of 1.2883707284927368
layers.5.0.weight has gradient mean of 1.610088586807251
layers.6.0.weight has gradient mean of 2.3182826042175293
layers.7.0.weight has gradient mean of 1.9781137704849243
layers.8.0.weight has gradient mean of 1.0172204971313477
layers.9.0.weight has gradient mean of 12.275727272033691
